# Φάση Γ: Classification Models

**Υπεύθυνος:** ML Engineer

**Μοντέλα:**
1. Random Forest
2. SVM


In [1]:
# Φάση Γ: Classification Models (Προσθήκη Μοντέλων για Imbalance & K-Means Features)
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import (RandomForestClassifier, LinearSVC, NaiveBayes, 
                                       MultilayerPerceptronClassifier, GBTClassifier, LogisticRegression)

print("Εκκίνηση SparkSession...")
spark = SparkSession.builder \
    .appName("Stroke_Classification_Augmented") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

# 1. ΦΟΡΤΩΣΗ ΤΩΝ ΕΜΠΛΟΥΤΙΣΜΕΝΩΝ ΔΕΔΟΜΕΝΩΝ (Από το K-Means)
print("Φόρτωση δεδομένων με Cluster Assignments...")
train_gold = spark.read.parquet("../data/train_gold_with_cluster.parquet")
test_gold = spark.read.parquet("../data/test_gold_with_cluster.parquet")

train_gold = train_gold.withColumn("stroke", train_gold["stroke"].cast(DoubleType()))
test_gold = test_gold.withColumn("stroke", test_gold["stroke"].cast(DoubleType()))

# 2. FEATURE AUGMENTATION: Ενσωμάτωση του Cluster στα Features
print("Ενσωμάτωση του K-Means Cluster στο Feature Vector...")
# Σιγουρευόμαστε ότι το cluster είναι Double για συμβατότητα
train_gold = train_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))
test_gold = test_gold.withColumn("cluster", F.col("cluster").cast(DoubleType()))

assembler = VectorAssembler(inputCols=["features", "cluster"], outputCol="augmented_features")

train_gold = assembler.transform(train_gold).drop("features").withColumnRenamed("augmented_features", "features")
test_gold = assembler.transform(test_gold).drop("features").withColumnRenamed("augmented_features", "features")

# Caching για δραστική μείωση του χρόνου εκπαίδευσης και αποφυγή memory leaks
train_gold.cache()
test_gold.cache()

# --- ADVANCED TECHNIQUE 1: Cost-Sensitive Learning (Βάρη Κλάσεων) ---
stroke_count = train_gold.filter(F.col("stroke") == 1.0).count()
total_count = train_gold.count()
balance_ratio = (total_count - stroke_count) / stroke_count
train_gold = train_gold.withColumn("class_weight", F.when(F.col("stroke") == 1.0, balance_ratio).otherwise(1.0))

# --- ADVANCED TECHNIQUE 2: Random Undersampling (Data-Centric Approach) ---
# Φτιάχνουμε ένα υποσύνολο όπου οι Υγιείς είναι ακριβώς όσοι και τα Εγκεφαλικά (50-50)
minority_df = train_gold.filter(F.col("stroke") == 1.0)
majority_df = train_gold.filter(F.col("stroke") == 0.0).sample(withReplacement=False, fraction=(stroke_count / (total_count - stroke_count)), seed=42)
train_undersampled = minority_df.union(majority_df)

# ==========================================
# 1. ΤΑ 4 ΑΡΧΙΚΑ ΣΟΥ ΜΟΝΤΕΛΑ
# ==========================================
print("Εκπαίδευση 4 Αρχικών Μοντέλων (Με χρήση Cluster Features)...")
rf = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=300, maxDepth=15, seed=22390225)
rf_preds = rf.fit(train_gold).transform(test_gold)

svm = LinearSVC(featuresCol="features", labelCol="stroke", maxIter=100)
svm_preds = svm.fit(train_gold).transform(test_gold)

nb = NaiveBayes(featuresCol="features", labelCol="stroke")
nb_preds = nb.fit(train_gold).transform(test_gold)

# Το input_size πλέον θα είναι +1 αυτόματα χάρη στον κώδικά σου!
input_size = len(train_gold.select("features").first()[0])
mlp = MultilayerPerceptronClassifier(maxIter=200, layers=[input_size, 32, 16, 2], seed=22390225, featuresCol="features", labelCol="stroke")
mlp_preds = mlp.fit(train_gold).transform(test_gold)


# ==========================================
# 2. ΤΑ ΝΕΑ ΜΟΝΤΕΛΑ (ΕΙΔΙΚΑ ΓΙΑ IMBALANCE)
# ==========================================
print("Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...")

# Νέο Μοντέλο 1: GBT (Gradient Boosted Trees)
gbt = GBTClassifier(featuresCol="features", labelCol="stroke", maxIter=150, maxDepth=5, seed=22390225)
gbt_preds = gbt.fit(train_gold).transform(test_gold)

# Νέο Μοντέλο 2: Logistic Regression (Με Class Weights)
lr = LogisticRegression(featuresCol="features", labelCol="stroke", weightCol="class_weight", maxIter=100)
lr_preds = lr.fit(train_gold).transform(test_gold)

# Νέο Μοντέλο 3: Random Forest εκπαιδευμένο στο 50-50 Undersampled Dataset
rf_under = RandomForestClassifier(featuresCol="features", labelCol="stroke", numTrees=300, maxDepth=15, seed=22390225)
rf_under_preds = rf_under.fit(train_undersampled).transform(test_gold)


# --- ΑΠΟΘΗΚΕΥΣΗ ΟΛΩΝ ---
print("Αποθήκευση προβλέψεων...")
rf_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_predictions.parquet")
svm_preds.select("stroke", "prediction", "rawPrediction").write.mode("overwrite").parquet("../data/svm_predictions.parquet")
nb_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/nb_predictions.parquet")
mlp_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/mlp_predictions.parquet")
# Νέα
gbt_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/gbt_predictions.parquet")
lr_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/lr_predictions.parquet")
rf_under_preds.select("stroke", "prediction", "probability").write.mode("overwrite").parquet("../data/rf_under_predictions.parquet")

spark.stop()
print("Η διαδικασία ολοκληρώθηκε επιτυχώς!")

Εκκίνηση SparkSession...


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/09 18:05:32 WARN Utils: Your hostname, cachyos-x8664, resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface enp4s0)
26/06/09 18:05:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/09 18:05:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/09 18:05:32 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Φόρτωση δεδομένων με Cluster Assignments...
Ενσωμάτωση του K-Means Cluster στο Feature Vector...
Εκπαίδευση 4 Αρχικών Μοντέλων (Με χρήση Cluster Features)...


26/06/09 18:05:39 WARN DAGScheduler: Broadcasting large task binary with size 1305.6 KiB
26/06/09 18:05:39 WARN DAGScheduler: Broadcasting large task binary with size 2.3 MiB
26/06/09 18:05:40 WARN DAGScheduler: Broadcasting large task binary with size 3.7 MiB
26/06/09 18:05:40 WARN DAGScheduler: Broadcasting large task binary with size 5.9 MiB
26/06/09 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 1048.1 KiB
26/06/09 18:05:41 WARN DAGScheduler: Broadcasting large task binary with size 8.5 MiB
26/06/09 18:05:42 WARN DAGScheduler: Broadcasting large task binary with size 1298.8 KiB
26/06/09 18:05:43 WARN DAGScheduler: Broadcasting large task binary with size 11.6 MiB
26/06/09 18:05:43 WARN DAGScheduler: Broadcasting large task binary with size 1481.6 KiB
26/06/09 18:05:44 WARN DAGScheduler: Broadcasting large task binary with size 15.0 MiB
26/06/09 18:05:45 WARN DAGScheduler: Broadcasting large task binary with size 1586.6 KiB
26/06/09 18:05:46 WARN DAGScheduler: 

Εκπαίδευση Νέων Μοντέλων (GBT, Logistic Regression, Undersampled RF)...


26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1000.7 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1001.2 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1002.2 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1003.0 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1005.3 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1008.0 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1008.5 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1009.5 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1010.3 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1012.7 KiB
26/06/09 18:06:39 WARN DAGScheduler: Broadcasting large task binary with size 1015.4 KiB
26/06/09 18:06:39 WAR

Αποθήκευση προβλέψεων...


26/06/09 18:07:13 WARN DAGScheduler: Broadcasting large task binary with size 14.8 MiB
26/06/09 18:07:14 WARN DAGScheduler: Broadcasting large task binary with size 1365.6 KiB
26/06/09 18:07:15 WARN DAGScheduler: Broadcasting large task binary with size 15.1 MiB


Η διαδικασία ολοκληρώθηκε επιτυχώς!
